In [33]:
import io
import json
import os
import sys


In [34]:
from contextlib import redirect_stdout

In [35]:
#!/usr/bin/env python3
"""
block-lanczos diagnostics for the 2x2 mock correlator.

this follows the block-lanczos recurrence with the candidate eq. 24 fix
that uses A1(t+2) in the leading term. passing the A2(0)=I test below
is evidence for that candidate, not a proof of the whole recurrence.
"""
# 1. imports and numerical precision
import hashlib
import json
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpmath import mp

mp.dps = 100
NSTR_DIGITS = 80
# 2. mock correlator construction
def build_mock(Nt=32):
    # build a 2x2 mock correlator C(t) from 16 hidden states
    r, Ns = 2, 16
    E, Zt, Z = [], [], []

    for k in range(Ns):
        Ek = mp.mpf("0.1") * (k + 1)
        if k % 2 == 0:
            z0 = mp.mpf("1.0")
            z1 = mp.mpf("-0.1") if k == 0 else mp.mpf("0.1")
        else:
            z0 = mp.mpf("0.1")
            z1 = mp.mpf("1.0")

        Zt.append((z0, z1))
        # normalise by sqrt(2 Ek) so C(t) looks like the usual spectral form
        Z.append((z0 / mp.sqrt(2 * Ek), z1 / mp.sqrt(2 * Ek)))
        E.append(Ek)

    C = []
    for t in range(Nt):
        Ct = mp.matrix(r, r)
        for k in range(Ns):
            ek = mp.exp(-E[k] * t)
            for a in range(r):
                for b in range(r):
                    Ct[a, b] += Z[k][a] * Z[k][b] * ek
        C.append(Ct)

    return C, E, Zt, Z

def print_mock(C, E, Zt, Z):
    # print the mock correlator and the overlap table
    print("[Eq 71-72] MOCK CORRELATOR CONSTRUCTION")
    print(f"  Nt       = {len(C)}")
    print(f"  N_states = {len(E)}")
    print()

    print("  k    E_k      Z~_{k,0}    Z~_{k,1}     Z_{k,0}         Z_{k,1}")
    for k in range(len(E)):
        print(
            f"  {k:2d}   {mp.nstr(E[k], 4):<8} "
            f"{mp.nstr(Zt[k][0], 4):<10} {mp.nstr(Zt[k][1], 4):<10} "
            f"{mp.nstr(Z[k][0], 10):<14} {mp.nstr(Z[k][1], 10)}"
        )
    print()

    print("  C(t) sample values:")
    sample_times = [t for t in [0, 1, 2, 5, 10, 15, 20, 31] if t < len(C)]
    for t in sample_times:
        print(
            f"  t={t:2d}:  C = [[{mp.nstr(C[t][0,0], 10)}, "
            f"{mp.nstr(C[t][0,1], 10)}],"
        )
        print(
            f"               [{mp.nstr(C[t][1,0], 10)}, "
            f"{mp.nstr(C[t][1,1], 10)}]]"
        )
# 3. matrix factorization and block-lanczos recursion
def eig_values_vectors(A):
    # wrap mpmath.eig so we get a plain list of eigenvalues plus the eigenvectors
    E, ER = mp.eig(A)
    try:
        vals = [E[k, 0] for k in range(E.rows)]
    except (TypeError, AttributeError):
        vals = list(E)

    if not hasattr(ER, "rows"):
        ER = mp.matrix(ER)

    return vals, ER
def factor_delta(Delta):
    # factor Delta = beta * gamma via Delta = V diag(lambda) V^-1
    # uh, this just assumes the square roots exist in whatever branch we picked
    # it does not check hermiticity or positive semidefiniteness of Delta
    r = Delta.rows

    if r == 1:
        s = mp.sqrt(Delta[0, 0])
        return mp.matrix([[s]]), mp.matrix([[s]])

    vals, V = eig_values_vectors(Delta)
    Vinv = mp.inverse(V)
    S = mp.diag([mp.sqrt(vals[k]) for k in range(r)])

    return V * S, S * Vinv


def block_lanczos(C, m):
    # run m steps of block lanczos on C
    Nt = len(C)
    r = C[0].rows

    # uh, A_j(t) shrinks by two time slices each step, so we need Nt >= 2m
    if Nt < 2 * m:
        raise ValueError(f"block_lanczos requires Nt >= 2m; got Nt={Nt}, m={m}")

    alphas = [None] * (m + 1)
    betas = [None] * (m + 1)
    gammas = [None] * (m + 1)
    As = [None] * (m + 1)
    Gs = [None] * (m + 1)
    Bs = [None] * (m + 1)

    # eq. 15: C(0) = beta_1 gamma_1
    b1, g1 = factor_delta(C[0])
    betas[1], gammas[1] = b1, g1
    ib1, ig1 = mp.inverse(b1), mp.inverse(g1)

    # eq. 20: A_1(t) = beta_1^-1 C(t) gamma_1^-1
    A1 = [ib1 * C[t] * ig1 for t in range(Nt)]
    As[1] = A1
    alphas[1] = A1[1]  # eq. 17

    if m == 1:
        return dict(alpha=alphas, beta=betas, gamma=gammas, A=As, G=Gs, B=Bs)

    a1 = alphas[1]

    # eq. 21
    Delta2 = A1[2] - a1 * a1
    b2, g2 = factor_delta(Delta2)
    betas[2], gammas[2] = b2, g2
    ib2, ig2 = mp.inverse(b2), mp.inverse(g2)

    # eqs. 22-23
    Gs[2] = [ib2 * (A1[t + 1] - a1 * A1[t]) for t in range(Nt - 1)]
    Bs[2] = [(A1[t + 1] - A1[t] * a1) * ig2 for t in range(Nt - 1)]

    # candidate eq. 24: leading term A_1(t+2)
    # the printed A_1(t+1) version gets tested separately below
    As[2] = [
        ib2
        * (A1[t + 2] + a1 * A1[t] * a1 - (a1 * A1[t + 1] + A1[t + 1] * a1))
        * ig2
        for t in range(Nt - 2)
    ]
    alphas[2] = As[2][1]  # eq. 29

    for j in range(2, m):
        Aj, Gj, Bj, Ajm1 = As[j], Gs[j], Bs[j], As[j - 1]
        aj, bj, gj = alphas[j], betas[j], gammas[j]

        # eq. 25
        Delta = Aj[2] - aj * aj - gj * bj
        bn, gn = factor_delta(Delta)
        betas[j + 1], gammas[j + 1] = bn, gn
        ibn, ign = mp.inverse(bn), mp.inverse(gn)

        # eqs. 26-27
        Gs[j + 1] = [
            ibn * (Aj[t + 1] - aj * Aj[t] - gj * Bj[t])
            for t in range(min(len(Aj) - 1, len(Bj)))
        ]
        Bs[j + 1] = [
            (Aj[t + 1] - Aj[t] * aj - Gj[t] * bj) * ign
            for t in range(min(len(Aj) - 1, len(Gj)))
        ]

        # eq. 28
        LA = min(len(Aj) - 2, len(Ajm1), len(Bj) - 1, len(Gj) - 1)
        As[j + 1] = []
        for t in range(LA):
            term = (
                Aj[t + 2]
                - (aj * Aj[t + 1] + Aj[t + 1] * aj)
                + aj * Aj[t] * aj
                + gj * Ajm1[t] * bj
                - (gj * Bj[t + 1] + Gj[t + 1] * bj)
                + gj * Bj[t] * aj
                + aj * Gj[t] * bj
            )
            As[j + 1].append(ibn * term * ign)

        alphas[j + 1] = As[j + 1][1]  # eq. 29

    return dict(alpha=alphas, beta=betas, gamma=gammas, A=As, G=Gs, B=Bs)



# 4. tridiagonal matrix construction and spectral reconstruction


def assemble_T(alphas, betas, gammas, m, r):
    # assemble the block-tridiagonal T^(m) from alpha, beta, gamma
    T = mp.matrix(m * r, m * r)

    for i in range(m):
        for a in range(r):
            for b in range(r):
                T[i * r + a, i * r + b] = alphas[i + 1][a, b]

    for i in range(m - 1):
        for a in range(r):
            for b in range(r):
                T[i * r + a, (i + 1) * r + b] = betas[i + 2][a, b]
                T[(i + 1) * r + a, i * r + b] = gammas[i + 2][a, b]

    return T


def submatrix(T, n):
    # grab the top-left n x n submatrix of T
    return mp.matrix([[T[i, j] for j in range(n)] for i in range(n)])


def ritz_energies(T):
    # diagonalise T and return ritz energies E = -log(lambda), sorted by real part
    # uh, sorting by real part is just a diagnostic convention
    # if T is non-hermitian the eigenvalues can be complex; we return them as computed
    vals, _ = eig_values_vectors(T)
    out = [-mp.log(v) for v in vals]
    out.sort(key=lambda x: mp.re(x))
    return out


def reconstruct_C(T, b1, g1, Nt, r):
    # reconstruct C(t) = beta_1 [T^t]_(1,1) gamma_1, eq. 60
    out, P = [], mp.eye(T.rows)

    for _ in range(Nt):
        block = mp.matrix([[P[a, b] for b in range(r)] for a in range(r)])
        out.append(b1 * block * g1)
        P = P * T

    return out


def maxdiff(A, B, r):
    # biggest element-wise difference between two r x r matrices
    return max(abs(A[a, b] - B[a, b]) for a in range(r) for b in range(r))


def matrix_max_abs(A, r):
    # biggest absolute matrix element
    return max(abs(A[a, b]) for a in range(r) for b in range(r))



# 5. core diagnostic tests


def run_diagnostics():
    # run the core block-lanczos diagnostic suite
    C, E, Zt, Z = build_mock(32)
    r = 2

    print_mock(C, E, Zt, Z)

    # eq. 15
    print()
    print("[Eq. 15] INITIAL FACTORIZATION  C(0) = β₁ γ₁")
    b1, g1 = factor_delta(C[0])
    init_err = maxdiff(b1 * g1, C[0], r)
    print(f"  β₁ = [[{mp.nstr(b1[0,0], 10)}, {mp.nstr(b1[0,1], 10)}],")
    print(f"        [{mp.nstr(b1[1,0], 10)}, {mp.nstr(b1[1,1], 10)}]]")
    print(f"  γ₁ = [[{mp.nstr(g1[0,0], 10)}, {mp.nstr(g1[0,1], 10)}],")
    print(f"        [{mp.nstr(g1[1,0], 10)}, {mp.nstr(g1[1,1], 10)}]]")
    print(f"  ‖β₁ γ₁ − C(0)‖_max = {mp.nstr(init_err, 3)}")

    m = 8
    out = block_lanczos(C, m)
    alphas, betas, gammas = out["alpha"], out["beta"], out["gamma"]
    As, Gs, Bs = out["A"], out["G"], out["B"]

    # eq. 17
    print()
    print("[Eq. 17] α₁ = β₁⁻¹ C(1) γ₁⁻¹")
    a1 = alphas[1]
    print(f"  α₁ = [[{mp.nstr(a1[0,0], 10)}, {mp.nstr(a1[0,1], 10)}],")
    print(f"        [{mp.nstr(a1[1,0], 10)}, {mp.nstr(a1[1,1], 10)}]]")

    # eq. 20
    print()
    print("[Eq. 20] A₁(t) = β₁⁻¹ C(t) γ₁⁻¹")
    for t in [0, 1, 2, 5, 10, 15]:
        A1t = As[1][t]
        print(f"  t={t:2d}: A₁ = [[{mp.nstr(A1t[0,0], 8)}, {mp.nstr(A1t[0,1], 8)}],")
        print(f"              [{mp.nstr(A1t[1,0], 8)}, {mp.nstr(A1t[1,1], 8)}]]")

    print(f"  ‖A₁(0) − I‖_max  = {mp.nstr(maxdiff(As[1][0], mp.eye(r), r), 3)}")
    print(f"  ‖A₁(1) − α₁‖_max = {mp.nstr(maxdiff(As[1][1], a1, r), 3)}")

    # eq. 21
    print()
    print("[Eq. 21] Δ₂ = A₁(2) − α₁ α₁ = β₂ γ₂")
    Delta2 = As[1][2] - a1 * a1
    print(f"  Δ₂ = [[{mp.nstr(Delta2[0,0], 10)}, {mp.nstr(Delta2[0,1], 10)}],")
    print(f"        [{mp.nstr(Delta2[1,0], 10)}, {mp.nstr(Delta2[1,1], 10)}]]")
    print(f"  ‖β₂ γ₂ − Δ₂‖_max = {mp.nstr(maxdiff(betas[2] * gammas[2], Delta2, r), 3)}")

    # eqs. 22-23
    print()
    print("[Eq. 22-23] G₂(t), B₂(t)")
    for t in [1, 2, 5, 10]:
        G2t, B2t = Gs[2][t], Bs[2][t]
        print(f"  t={t:2d}: G₂ = [[{mp.nstr(G2t[0,0], 8)}, {mp.nstr(G2t[0,1], 8)}],"
              f" [{mp.nstr(G2t[1,0], 8)}, {mp.nstr(G2t[1,1], 8)}]]")
        print(f"        B₂ = [[{mp.nstr(B2t[0,0], 8)}, {mp.nstr(B2t[0,1], 8)}],"
              f" [{mp.nstr(B2t[1,0], 8)}, {mp.nstr(B2t[1,1], 8)}]]")

    # candidate eq. 24
    print()
    print("=" * 72)
    print("[Eq. 24 CANDIDATE] A₂(t) with A₁(t+2) leading term")
    print("=" * 72)
    for t in [0, 1, 2, 5]:
        A2t = As[2][t]
        print(f"  t={t:2d}: A₂ = [[{mp.nstr(A2t[0,0], 8)}, {mp.nstr(A2t[0,1], 8)}],"
              f" [{mp.nstr(A2t[1,0], 8)}, {mp.nstr(A2t[1,1], 8)}]]")

    a2_err = maxdiff(As[2][0], mp.eye(r), r)
    print(f"  ‖A₂(0) − I‖_max = {mp.nstr(a2_err, 3)}")
    print("  This identity check tests one consequence of the candidate recurrence;")
    print("  it does not prove the full recurrence.")

    # falsification of printed eq. 24
    print()
    print("[FALSIFICATION] PRINTED Eq. 24 (A₁(t+1) leading term) must fail")
    ib2, ig2 = mp.inverse(betas[2]), mp.inverse(gammas[2])
    A2_printed = (
        ib2
        * (As[1][1] + a1 * As[1][0] * a1 - (a1 * As[1][1] + As[1][1] * a1))
        * ig2
    )
    err_p = maxdiff(A2_printed, mp.eye(r), r)

    print(f"  Printed form  : ‖A₂(0) − I‖_max = {mp.nstr(err_p, 5)}")
    print(f"  Candidate form: ‖A₂(0) − I‖_max = {mp.nstr(a2_err, 3)}")

    # eqs. 25-29 loop
    print()
    print("[Eq. 25-29] LOOP j = 3..8")
    print(f"  j    ‖β_j γ_j − Δ_j‖       ‖α_j‖       ‖β_j‖       ‖γ_j‖")
    for j in range(2, m + 1):
        if j > 2:
            Delta_j = (
                As[j - 1][2]
                - alphas[j - 1] * alphas[j - 1]
                - gammas[j - 1] * betas[j - 1]
            )
            consistency = mp.nstr(maxdiff(betas[j] * gammas[j], Delta_j, r), 3)
        else:
            consistency = "—"

        aj, bj, gj = alphas[j], betas[j], gammas[j]
        nA = mp.nstr(matrix_max_abs(aj, r), 4)
        nB = mp.nstr(matrix_max_abs(bj, r), 4)
        nG = mp.nstr(matrix_max_abs(gj, r), 4)
        print(f"  {j:2d}   {consistency:<18} {nA:<10} {nB:<10} {nG}")

    # eq. 30
    print()
    print("[Eq. 30] CONSISTENCY  G_j(1) = γ_j,  B_j(1) = β_j")
    print(f"  j    ‖G_j(1) − γ_j‖       ‖B_j(1) − β_j‖")
    worst = mp.mpf(0)
    for j in range(2, m + 1):
        eg = maxdiff(Gs[j][1], gammas[j], r)
        eb = maxdiff(Bs[j][1], betas[j], r)
        worst = max(worst, eg, eb)
        print(f"  {j:2d}   {mp.nstr(eg, 3):<18} {mp.nstr(eb, 3)}")
    print(f"  WORST = {mp.nstr(worst, 3)}")

    # eq. 32
    T8 = assemble_T(alphas, betas, gammas, m, r)
    print()
    print("[Eq. 32] T^(8) — 16×16 block tridiagonal")
    print("  Block structure (j-row × i-col): α diag, β upper, γ lower")
    for i in range(m):
        line = "  "
        for j in range(m):
            if i == j:
                line += " α "
            elif i == j - 1:
                line += " γ "
            elif i == j + 1:
                line += " β "
            else:
                line += " . "
        print(line)

    print("  Diagonal blocks α_j:")
    for j in range(1, m + 1):
        aj = alphas[j]
        print(f"    α_{j} = [[{mp.nstr(aj[0,0], 8)}, {mp.nstr(aj[0,1], 8)}],"
              f" [{mp.nstr(aj[1,0], 8)}, {mp.nstr(aj[1,1], 8)}]]")

    # eqs. 36-37
    print()
    print("[Eq. 36-37] RITZ DECOMPOSITION of T^(8) → λ_k → E_k = -ln λ_k")
    Er = ritz_energies(T8)
    print(f"  k    λ_k                E_k^(8)                   E_k true   |err|")
    all_ok = True
    for k in range(16):
        lam = mp.e ** (-Er[k])
        err = abs(mp.re(Er[k]) - E[k])
        all_ok &= (err < mp.mpf("1e-50"))
        print(f"  {k:2d}   {mp.nstr(mp.re(lam), 12):<18}"
              f"{mp.nstr(mp.re(Er[k]), 20):<24}"
              f"{mp.nstr(E[k], 4):<10}{mp.nstr(err, 3)}")
    print(f"  All |E_k − E_k^true| < 1e-50 : {all_ok}")

    # eq. 60
    print()
    print("[Eq. 60] RECONSTRUCTION  C(t) = β₁ [T^(8)^t]_{(1,1)} γ₁")
    Crec = reconstruct_C(T8, betas[1], gammas[1], 32, r)
    print(f"  t    ‖C_recon(t) − C(t)‖_max")
    for t in range(16):
        print(f"  {t:2d}   {mp.nstr(maxdiff(Crec[t], C[t], r), 3)}")
    rec_err = max(maxdiff(Crec[t], C[t], r) for t in range(16))

    # scalar lanczos on the diagonals
    print()
    print("[Scalar m=16] LANCZOS ON DIAGONAL C_00(t) AND C_11(t)")
    for d in (0, 1):
        Cd = [mp.matrix([[C[t][d, d]]]) for t in range(32)]
        o = block_lanczos(Cd, 16)
        Td = assemble_T(o["alpha"], o["beta"], o["gamma"], 16, 1)
        Ed = ritz_energies(Td)
        print(f"  C_{d}{d}(t):")
        print(f"    E_0^(16) = {mp.nstr(mp.re(Ed[0]), 18)}  (true 0.1)  "
              f"|err| = {mp.nstr(abs(mp.re(Ed[0]) - E[0]), 3)}")
        print(f"    E_1^(16) = {mp.nstr(mp.re(Ed[1]), 18)}  (true 0.2)  "
              f"|err| = {mp.nstr(abs(mp.re(Ed[1]) - E[1]), 3)}")
        Cr = reconstruct_C(Td, o["beta"][1], o["gamma"][1], 32, 1)
        rerr = max(abs(Cr[t][0, 0] - Cd[t][0, 0]) for t in range(32))
        print(f"    Eq. 60 reconstruction (t≤31): max err = {mp.nstr(rerr, 3)}")

    # summary
    print()
    print("SUMMARY")
    eq30_ok = float(worst) < 1e-80
    a2_ok = float(a2_err) < 1e-80
    printed_fails = float(err_p) > 1e-3
    init_ok = float(init_err) < 1e-80
    rec_ok = float(rec_err) < 1e-50

    print(f"  Eq. 15  β₁ γ₁ = C(0)              : "
          f"{'PASS' if init_ok else 'FAIL'}  ({mp.nstr(init_err, 3)})")
    print(f"  Eq. 30  G_j(1)=γ_j, B_j(1)=β_j    : "
          f"{'PASS' if eq30_ok else 'FAIL'}  (worst {mp.nstr(worst, 3)})")
    print(f"  Eq. 24  A₂(0) = I (candidate)     : "
          f"{'PASS' if a2_ok else 'FAIL'}  ({mp.nstr(a2_err, 3)})")
    print(f"  Eq. 24  Printed form fails A₂(0)=I: "
          f"{'PASS' if printed_fails else 'FAIL'}  (printed gives {mp.nstr(err_p, 3)})")
    print(f"  Eq. 37  Block spectrum m=8        : "
          f"{'PASS' if all_ok else 'FAIL'}")
    print(f"  Eq. 60  Reconstruction            : "
          f"{'PASS' if rec_ok else 'FAIL'}  ({mp.nstr(rec_err, 3)})")



# 6. figure 2 reproduction and CSV export


def extract_all(C, mmax, r):
    # return ritz data for every m = 1..mmax
    out = block_lanczos(C, mmax)
    T_full = assemble_T(out["alpha"], out["beta"], out["gamma"], mmax, r)

    history = []
    for m in range(1, mmax + 1):
        T_m = submatrix(T_full, m * r)
        E_vals = ritz_energies(T_m)
        lambdas = [mp.e ** (-E) for E in E_vals]
        history.append(dict(m=m, t=2 * m - 1, lambdas=lambdas, energies=E_vals))
    return history


def print_history(method, history, E_true):
    # print all ritz values for one method
    print(f"[{method}] all Ritz values per iteration")
    for entry in history:
        m, t = entry["m"], entry["t"]
        E_vals = entry["energies"]
        print(f"  m={m:2d}  t=2m-1={t:2d}   ({len(E_vals)} eigenvalues)")
        for k, E_k in enumerate(E_vals):
            true_val = E_true[k] if k < len(E_true) else None
            err = (
                f"|err|={mp.nstr(abs(mp.re(E_k) - true_val), 3)}"
                if true_val is not None else "—"
            )
            tv = mp.nstr(true_val, 4) if true_val is not None else "—"
            print(f"    rank {k:2d}: λ = {mp.nstr(mp.re(entry['lambdas'][k]), 8):<14}"
                  f"E^(m) = {mp.nstr(mp.re(E_k), 18):<24} "
                  f"true = {tv:<6} {err}")
    print()


def plot_fig2(all_data, png="fig2_reproduction.png"):
    # reproduce fig. 2 and report data-integrity checks
    def _snapshot(data):
        return {
            m: [float(mp.re(Ek)) for e in data[m] for Ek in e["energies"]]
            for m in ("block", "scalar_C00", "scalar_C11")
        }

    snap_before = _snapshot(all_data)
    hash_before = {
        m: hashlib.sha256(json.dumps(v).encode()).hexdigest()[:16]
        for m, v in snap_before.items()
    }

    csv_path = "fig2_data.csv"
    csv_hash_before = None
    if os.path.exists(csv_path):
        with open(csv_path, "rb") as f:
            csv_hash_before = hashlib.sha256(f.read()).hexdigest()[:16]

    plt.rcParams.update({
        "mathtext.fontset": "cm",
        "font.family": "serif",
        "font.size": 9,
        "axes.linewidth": 0.6,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "xtick.direction": "in",
        "ytick.direction": "in",
    })

    fig, ax = plt.subplots(figsize=(8.5, 2.8))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    for k in range(16):
        ax.axhline(0.1 * (k + 1), color="black", lw=0.6, zorder=1)

    styles = {
        "block": ("tab:blue", r"$C$"),
        "scalar_C00": ("tab:orange", r"$C_{00}$"),
        "scalar_C11": ("tab:red", r"$C_{11}$"),
    }

    for method, (col, _lab) in styles.items():
        hist = all_data[method]
        xs0, ys0, xs1, ys1, xsh, ysh = [], [], [], [], [], []
        for e in hist:
            t = e["t"]
            E_vals = e["energies"]
            if len(E_vals) >= 1:
                xs0.append(t)
                ys0.append(float(mp.re(E_vals[0])))
            if len(E_vals) >= 2:
                xs1.append(t)
                ys1.append(float(mp.re(E_vals[1])))
            for k in range(2, len(E_vals)):
                xsh.append(t)
                ysh.append(float(mp.re(E_vals[k])))

        if xsh:
            ax.plot(xsh, ysh, linestyle="None", marker="o", markersize=5.0,
                    color=col, markeredgecolor="white", markeredgewidth=1.0,
                    alpha=1.0, zorder=2)
        if xs0:
            ax.plot(xs0, ys0, "-", color=col, lw=1.0, marker="o", markersize=5.0,
                    markeredgecolor="white", markeredgewidth=1.0, zorder=3)
        if xs1:
            ax.plot(xs1, ys1, "--", color=col, lw=1.0, marker="o", markersize=5.0,
                    markeredgecolor="white", markeredgewidth=1.0, zorder=3)

    handles = [Line2D([0], [0], color=col, lw=1.2, label=lab)
               for (col, lab) in styles.values()]
    ax.legend(handles=handles, loc="upper left", frameon=False, fontsize=10,
              handlelength=1.2, borderpad=0.3, labelspacing=0.3)

    ax.set_xlim(-0.5, 32)
    ax.set_ylim(0.05, 1.65)
    ax.set_xticks([0, 5, 10, 15, 20, 25, 30])
    ax.set_yticks([0.5, 1.0, 1.5])
    ax.set_xlabel(r"$t = 2m-1$", fontsize=11)
    ax.set_ylabel(r"$E_k^{(m)} = -\log\,\lambda_k^{(m)}$", fontsize=11)
    ax.grid(False)
    plt.tight_layout(pad=0.3)
    plt.savefig(png, dpi=300, bbox_inches="tight")
    plt.close(fig)

    snap_after = _snapshot(all_data)
    hash_after = {
        m: hashlib.sha256(json.dumps(v).encode()).hexdigest()[:16]
        for m, v in snap_after.items()
    }

    csv_hash_after = None
    if os.path.exists(csv_path):
        with open(csv_path, "rb") as f:
            csv_hash_after = hashlib.sha256(f.read()).hexdigest()[:16]

    print("fig. 2 — point counts and data integrity")
    print(f"  {'method':<12}{'rank0':>8}{'rank1':>8}{'rank>=2':>10}{'total':>8}")
    for method in ("block", "scalar_C00", "scalar_C11"):
        hist = all_data[method]
        n0 = sum(1 for e in hist if len(e["energies"]) >= 1)
        n1 = sum(1 for e in hist if len(e["energies"]) >= 2)
        nh = sum(max(0, len(e["energies"]) - 2) for e in hist)
        nt = sum(len(e["energies"]) for e in hist)
        print(f"  {method:<12}{n0:>8d}{n1:>8d}{nh:>10d}{nt:>8d}")

    print(f"\n  all_data hash (before) : {hash_before}")
    print(f"  all_data hash (after)  : {hash_after}")
    print(f"  all_data unchanged     : {hash_before == hash_after}")

    if csv_hash_before is not None:
        print(f"\n  csv hash (before)      : {csv_hash_before}")
        print(f"  csv hash (after)       : {csv_hash_after}")
        print(f"  csv unchanged          : {csv_hash_before == csv_hash_after}")
    else:
        print(f"\n  csv file not found at '{csv_path}' — nothing to compare")

    print(f"\n  saved {png}")


def run_figure():
    # build fig. 2, export the csv, and run caption checks
    C, E, _Zt, _Z = build_mock(32)
    print("fig. 2 reproduction")
    print(f"  mp.dps  = {mp.dps}   nstr_digits = {NSTR_DIGITS}")
    print(f"  nt      = {len(C)}   states = {len(E)}   x-axis = t = 2m-1")

    block_hist = extract_all(C, 8, 2)
    print_history("block lanczos (r=2) on full 2x2 c(t)", block_hist, E)
    print()

    c00_hist = extract_all([mp.matrix([[C[t][0, 0]]]) for t in range(32)], 16, 1)
    print_history("scalar lanczos on c_00(t)", c00_hist, E)
    print()

    c11_hist = extract_all([mp.matrix([[C[t][1, 1]]]) for t in range(32)], 16, 1)
    print_history("scalar lanczos on c_11(t)", c11_hist, E)

    all_data = {"block": block_hist, "scalar_C00": c00_hist, "scalar_C11": c11_hist}

    print("building fig2_data.csv "
          "(lossless: mp.nstr with {} sig figs)".format(NSTR_DIGITS))
    rows = []
    for method, hist in all_data.items():
        for entry in hist:
            for rank, E_k in enumerate(entry["energies"]):
                rows.append(dict(
                    method=method,
                    m=entry["m"],
                    t=entry["t"],
                    energy_rank=rank,
                    lambda_real=mp.nstr(mp.re(entry["lambdas"][rank]), NSTR_DIGITS),
                    extracted_energy=mp.nstr(mp.re(E_k), NSTR_DIGITS),
                    true_energy=(
                        mp.nstr(E[rank], NSTR_DIGITS)
                        if rank < len(E) else "nan"
                    ),
                ))

    df = pd.DataFrame(rows)
    df.to_csv("fig2_data.csv", index=False)
    print(f"  {len(df)} rows written (string columns, round-trip lossless)")
    for method in ["block", "scalar_C00", "scalar_C11"]:
        s = df[df.method == method]
        print(f"    {method:<12} : {len(s)} rows, "
              f"m={s.m.min()}..{s.m.max()}, max rank={s.energy_rank.max()}")

    plot_fig2(all_data)

    print()
    print("paper caption check")
    checks = [
        ("block m=8   e_0", block_hist[7]["energies"][0], mp.mpf("0.1")),
        ("block m=8   e_1", block_hist[7]["energies"][1], mp.mpf("0.2")),
        ("c_00  m=16  e_0", c00_hist[15]["energies"][0], mp.mpf("0.1")),
        ("c_00  m=16  e_1", c00_hist[15]["energies"][1], mp.mpf("0.2")),
        ("c_11  m=16  e_0", c11_hist[15]["energies"][0], mp.mpf("0.1")),
        ("c_11  m=16  e_1", c11_hist[15]["energies"][1], mp.mpf("0.2")),
    ]
    for name, val_mp, target in checks:
        diff = abs(mp.re(val_mp) - target)
        ok = diff < mp.mpf("1e-50")
        print(f"  {name} = {mp.nstr(mp.re(val_mp), 20)}   "
              f"|δ| = {mp.nstr(diff, 3)}   {'pass' if ok else 'fail'}")

    return all_data, df



# 7. higher-rank convergence analysis


def _read_csv_lossless(csv_path):
    # read the csv while keeping the high-precision string columns
    df = pd.read_csv(
        csv_path,
        dtype={
            "extracted_energy": str,
            "true_energy": str,
            "lambda_real": str,
        },
    ).sort_values(["method", "m", "energy_rank"]).reset_index(drop=True)

    for col in ["m", "t", "energy_rank"]:
        df[col] = df[col].astype(int)

    df["extracted_energy"] = df["extracted_energy"].map(lambda x: mp.mpf(x))
    df["true_energy"] = df["true_energy"].map(
        lambda x: mp.mpf(x) if x != "nan" else mp.nan
    )
    return df


def _nearest_true_index(E_value, TRUE_E):
    return min(range(len(TRUE_E)), key=lambda j: abs(TRUE_E[j] - E_value))


def run_higher_rank_check(csv_path="fig2_data.csv"):
    # check saturation and higher-rank convergence against the paper claims
    TOL_SAT = mp.mpf("1e-50")
    TOL_DEGEN = mp.mpf("1e-3")
    TRUE_E = [mp.mpf("0.1") * (k + 1) for k in range(16)]

    df = _read_csv_lossless(csv_path)

    print("higher-rank eigenvalue check vs. paper claims")
    print(f"  loaded         : {len(df)} rows from {csv_path} (lossless read)")
    print(f"  true spectrum  : e_k = 0.1(k+1), k = 0..15")
    print(f"  tol_sat        : {mp.nstr(TOL_SAT, 3)}")
    print(f"  tol_degen      : {mp.nstr(TOL_DEGEN, 3)}")

    print("part 1 — saturation: full spectrum at the paper's claimed step")
    sat = {"block": 8, "scalar_C00": 16, "scalar_C11": 16}
    paper_claim = {
        "block": "solves for the true eigenvalues exactly after 8 steps",
        "scalar_C00": "requires 16 steps",
        "scalar_C11": "requires 16 steps",
    }

    for method in ["block", "scalar_C00", "scalar_C11"]:
        m_sat = sat[method]
        s = df[(df.method == method) & (df.m == m_sat)].sort_values("energy_rank")
        E = s.extracted_energy.values
        ranks = s.energy_rank.values
        t = int(s.t.values[0])

        print(f"\n  [{method}]  m={m_sat}, x=t={t}")
        print(f"  paper claim: '{paper_claim[method]}'")
        print(f"  {'rank':>4}  {'E_k^(m)':>26}  {'E_k^true':>8}  "
              f"{'|δ|':>11}  {'nearest':>7}  match")

        all_match = True
        bijection = True
        for k in range(len(E)):
            r = int(ranks[k])
            ni = _nearest_true_index(E[k], TRUE_E)
            d_true = abs(E[k] - TRUE_E[r])
            ok = (d_true < TOL_SAT) and (ni == r)
            all_match &= ok
            bijection &= (ni == r)
            print(f"  {r:>4}  {mp.nstr(E[k], 25):>26}  "
                  f"{mp.nstr(TRUE_E[r], 4):>8}  "
                  f"{mp.nstr(d_true, 3):>11}  "
                  f"{mp.nstr(TRUE_E[ni], 4):>7}  "
                  f"{'ok' if ok else 'fail'}")

        print(f"  all 16 ranks within {mp.nstr(TOL_SAT, 3)} : {all_match}")
        print(f"  rank k nearest to true level k for all k   : {bijection}")

    print("part 2 — per-rank convergence (all 16 ranks, every method)")
    print("  table entry = |e_k^(m) − e_k^true|   ('.' = converged below 1e-10)")
    for method in ["block", "scalar_C00", "scalar_C11"]:
        sub = df[df.method == method]
        mmax = int(sub.m.max())
        rmax = int(sub.energy_rank.max())
        print(f"\n  [{method}]   m = 1 .. {mmax}")
        header = "  rank | " + " ".join(f"m={m:<2}" for m in range(1, mmax + 1))
        print(header)

        violations = []
        for r in range(rmax + 1):
            line = f"  {r:>4} | "
            prev_delta = None
            for m in range(1, mmax + 1):
                row = sub[(sub.m == m) & (sub.energy_rank == r)]
                if row.empty:
                    line += " .   "
                    prev_delta = None
                    continue

                E = row.extracted_energy.values[0]
                delta = abs(E - TRUE_E[r])

                if delta < mp.mpf("1e-10"):
                    line += "  .  "
                elif delta >= 1000:
                    line += f"{mp.nstr(delta, 2):<5}"
                elif delta >= 1:
                    line += f"{mp.nstr(delta, 3):<5}"
                else:
                    line += f"{mp.nstr(delta, 2):<5}"

                if prev_delta is not None and delta > prev_delta + mp.mpf("1e-8"):
                    violations.append((r, m - 1, m, prev_delta, delta))
                prev_delta = delta
            print(line)

        if violations:
            print(f"  non-monotone in {len(violations)} steps (expected for oblique):")
            for r, m1, m2, d1, d2 in violations[:5]:
                print(f"    rank {r}: m={m1} δ={mp.nstr(d1, 3)} → "
                      f"m={m2} δ={mp.nstr(d2, 3)}")
        else:
            print("  convergence from above (monotone) holds for all ranks")

    print("part 3 — within-m non-degeneracy")
    print(f"  distinct eigenvalues at same m should be > {mp.nstr(TOL_DEGEN, 3)} apart.")
    any_degen = False
    for method in ["block", "scalar_C00", "scalar_C11"]:
        sub = df[df.method == method]
        worst_gap, worst_m = mp.inf, None
        for m in sorted(sub.m.unique()):
            E = sub[sub.m == m].sort_values("energy_rank").extracted_energy.values
            if len(E) < 2:
                continue
            gaps = [abs(E[i + 1] - E[i]) for i in range(len(E) - 1)]
            g = min(gaps)
            if g < worst_gap:
                worst_gap, worst_m = g, m
            if g < TOL_DEGEN:
                any_degen = True
                print(f"  [{method}] m={m}: min gap = {mp.nstr(g, 3)}  < "
                      f"{mp.nstr(TOL_DEGEN, 3)}   ← degenerate")
        print(f"  [{method}]  worst gap over all m: {mp.nstr(worst_gap, 3)} "
              f"at m={worst_m}   "
              f"{'ok' if worst_gap >= TOL_DEGEN else 'degenerate'}")

    print(f"\n  any near-degeneracy anywhere : {any_degen}")

    print("verdict")
    def sat_ok(method):
        m_sat = sat[method]
        s = df[(df.method == method) & (df.m == m_sat)].sort_values("energy_rank")
        E = s.extracted_energy.values
        return all(abs(E[k] - TRUE_E[k]) < TOL_SAT for k in range(len(E)))

    print(f"  block m=8, all 16 ranks within {mp.nstr(TOL_SAT, 3)}  : "
          f"{'pass' if sat_ok('block') else 'fail'}")
    print(f"  scalar c_00 m=16, all 16 ranks within {mp.nstr(TOL_SAT, 3)} : "
          f"{'pass' if sat_ok('scalar_C00') else 'fail'}")
    print(f"  scalar c_11 m=16, all 16 ranks within {mp.nstr(TOL_SAT, 3)} : "
          f"{'pass' if sat_ok('scalar_C11') else 'fail'}")
    print(f"  non-degenerate at every m (gap > {mp.nstr(TOL_DEGEN, 3)})   : "
          f"{'pass' if not any_degen else 'fail'}")
    print()



# 8. eigenvalue-ordering diagnostics


def run_ordering_diagnostic(csv_path="fig2_data.csv"):
    # print per-iteration ordering, nearest-level, and locking diagnostics
    TOL_CONV = mp.mpf("1e-6")
    TOL_GAP = mp.mpf("1e-10")
    df = _read_csv_lossless(csv_path)
    Etrue = [mp.mpf("0.1") * (k + 1) for k in range(16)]

    print("ordering diagnostic — every eigenvalue, every rank, every iteration")
    print(f"loaded   : {len(df)} rows from {csv_path} (lossless)")
    print(f"true e_n : 0.1(n+1), n = 0..15")
    print(f"tol_conv = {mp.nstr(TOL_CONV, 3)}   tol_gap = {mp.nstr(TOL_GAP, 3)}")
    print()
    print("columns:")
    print("  rank     : 0 = lowest extracted re e, 1 = 2nd lowest, ...")
    print("  near     : index n of nearest true level e_n")
    print("  |δ_near| : |e − e_near|")
    print("  true_r   : e_rank  (rank-matched true level)")
    print("  |δ_rank| : |e − e_rank|")
    print("  *        : nearest true level ≠ rank  (still converging)")
    print("  sorted   : extracted energies strictly increasing in rank")

    summary = {}

    for method in ["block", "scalar_C00", "scalar_C11"]:
        sub = df[df.method == method]
        mmax = int(sub.m.max())
        print(f"# method: {method}   (m = 1..{mmax})")
        print("\n[part a] per-iteration table")

        hdr = (f"{'m':>3}{'t':>5}  {'rank':>4}  {'E_k^(m)':>22}  "
               f"{'near':>5}  {'|δ_near|':>11}  {'true_r':>7}  "
               f"{'|δ_rank|':>11}  sorted")
        print("-" * len(hdr))
        print(hdr)
        print("-" * len(hdr))

        lock_on = {}
        any_unsorted = False

        for m in sorted(sub.m.unique()):
            s = sub[sub.m == m].sort_values("energy_rank")
            E = s.extracted_energy.values
            rk = s.energy_rank.values
            t = int(s.t.values[0])
            is_sorted = all(E[i + 1] > E[i] for i in range(len(E) - 1))
            if not is_sorted:
                any_unsorted = True

            for k, E_k in enumerate(E):
                r = int(rk[k])
                ni = _nearest_true_index(E_k, Etrue)
                dn = abs(Etrue[ni] - E_k)
                Er = Etrue[r] if r < len(Etrue) else mp.nan
                dr = abs(E_k - Er)

                if dr < TOL_CONV and r not in lock_on:
                    lock_on[r] = m

                mark = "" if ni == r else "*"
                print(f"{m:>3}{t:>5}  {r:>4}  {mp.nstr(E_k, 20):>22}  "
                      f"{ni:>4}{mark}  {mp.nstr(dn, 3):>11}  "
                      f"{mp.nstr(Er, 3):>7}  {mp.nstr(dr, 3):>11}  {is_sorted}")

            gaps = [abs(E[i + 1] - E[i]) for i in range(len(E) - 1)]
            if len(gaps) > 0 and min(gaps) < TOL_GAP:
                print(f"    !!! m={m}: min gap = {mp.nstr(min(gaps), 3)} "
                      f"< {mp.nstr(TOL_GAP, 3)}")
            print("-" * len(hdr))

        print("\n[part b] per-rank trajectory: nearest true index vs m")
        print("  (no mark → nearest == rank, * → nearest ≠ rank, . → rank absent)")
        rank_max = int(sub.energy_rank.max())
        print(f"\n  {'rank':>4}  " +
              "".join(f"m={m:<3}" for m in range(1, mmax + 1)))

        for r in range(rank_max + 1):
            line = f"  {r:>4}  "
            for m in range(1, mmax + 1):
                row = sub[(sub.m == m) & (sub.energy_rank == r)]
                if row.empty:
                    line += ".    "
                else:
                    E_k = row.extracted_energy.values[0]
                    ni = _nearest_true_index(E_k, Etrue)
                    tag = f"{ni}" if ni == r else f"{ni}*"
                    line += f"{tag:<5}"
            print(line)

        print(f"\n[part c] first m with |e_rank − e_rank^true| < {mp.nstr(TOL_CONV, 3)}")
        for r in range(rank_max + 1):
            cm = lock_on.get(r, None)
            status = f"m = {cm}" if cm is not None else "never within tol"
            print(f"  rank {r:>2}: {status}")

        print(f"\n[part d] at m = {mmax} (saturation): rank r maps to true level r?")
        s_sat = sub[sub.m == mmax].sort_values("energy_rank")
        E_sat = s_sat.extracted_energy.values
        all_ok = True
        for k in range(len(E_sat)):
            ni = _nearest_true_index(E_sat[k], Etrue)
            if ni != k:
                print(f"  rank {k:>2}: nearest = {ni}   mismatch  "
                      f"(e = {mp.nstr(E_sat[k], 12)}, "
                      f"nearest true = {mp.nstr(Etrue[ni], 3)})")
                all_ok = False
        if all_ok:
            print(f"  all {len(E_sat)} ranks map correctly to true levels  →  ok")

        summary[method] = dict(
            mmax=mmax,
            n_at_sat=len(E_sat),
            bijection=all_ok,
            lock_on=lock_on,
            unsorted=any_unsorted,
        )

    print("global summary")
    print(f"  {'method':<14}{'n at sat':>10}{'bijection':>12}"
          f"{'ranks converged':>18}{'sorted throughout':>20}")
    for method in ["block", "scalar_C00", "scalar_C11"]:
        s = summary[method]
        nconv = len(s["lock_on"])
        print(f"  {method:<14}{s['n_at_sat']:>10}"
              f"{str(s['bijection']):>12}"
              f"{nconv:>10} / 16"
              f"{str(not s['unsorted']):>20}")



# 9. main entry point


OUTPUT_DIR = "output"


def capture_and_show_summary(func, filepath, delimiter):
    """Run func with stdout redirected to filepath; print only the summary."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    buf = io.StringIO()
    with redirect_stdout(buf):
        func()
    text = buf.getvalue()

    with open(filepath, "w") as f:
        f.write(text)

    n_lines = text.count("\n")
    print(f"\n[{os.path.basename(filepath)}]  {n_lines} lines  →  {filepath}")

    if delimiter and delimiter in text:
        summary = text.split(delimiter, 1)[1]
        print(f"\n{delimiter}{summary}")


def write_ritz_json():
    """Dump every Ritz value to a structured JSON file."""
    C, E, _Zt, _Z = build_mock(32)
    histories = {
        "block":      extract_all(C, 8, 2),
        "scalar_C00": extract_all([mp.matrix([[C[t][0, 0]]]) for t in range(32)], 16, 1),
        "scalar_C11": extract_all([mp.matrix([[C[t][1, 1]]]) for t in range(32)], 16, 1),
    }

    data = {}
    for name, hist in histories.items():
        data[name] = [
            {
                "m": e["m"],
                "t": e["t"],
                "energies": [str(mp.re(v)) for v in e["energies"]],
                "lambdas":  [str(mp.re(v)) for v in e["lambdas"]],
            }
            for e in hist
        ]

    with open(f"{OUTPUT_DIR}/ritz_values.json", "w") as f:
        json.dump(data, f, indent=2)

    print(f"\n[ritz_values.json]  written  →  {OUTPUT_DIR}/ritz_values.json")


def main():
    capture_and_show_summary(
        run_diagnostics,
        f"{OUTPUT_DIR}/diagnostic.txt",
        "SUMMARY",
    )
    capture_and_show_summary(
        run_figure,
        f"{OUTPUT_DIR}/fig2_full.txt",
        "paper caption check",
    )
    capture_and_show_summary(
        run_higher_rank_check,
        f"{OUTPUT_DIR}/higher_rank.txt",
        "verdict",
    )
    capture_and_show_summary(
        run_ordering_diagnostic,
        f"{OUTPUT_DIR}/ordering.txt",
        "global summary",
    )

    write_ritz_json()
    print(f"all verbose logs in {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()


[diagnostic.txt]  194 lines  →  output/diagnostic.txt

SUMMARY
  Eq. 15  β₁ γ₁ = C(0)              : PASS  (5.71e-101)
  Eq. 30  G_j(1)=γ_j, B_j(1)=β_j    : PASS  (worst 1.43e-101)
  Eq. 24  A₂(0) = I (candidate)     : PASS  (1.14e-100)
  Eq. 24  Printed form fails A₂(0)=I: PASS  (printed gives 4.47)
  Eq. 37  Block spectrum m=8        : PASS
  Eq. 60  Reconstruction            : PASS  (2.0e-100)


[fig2_full.txt]  423 lines  →  output/fig2_full.txt

paper caption check
  block m=8   e_0 = 0.1   |δ| = 1.0e-94   pass
  block m=8   e_1 = 0.2   |δ| = 3.01e-94   pass
  c_00  m=16  e_0 = 0.1   |δ| = 4.0e-86   pass
  c_00  m=16  e_1 = 0.2   |δ| = 1.02e-80   pass
  c_11  m=16  e_0 = 0.1   |δ| = 3.3e-85   pass
  c_11  m=16  e_1 = 0.2   |δ| = 8.43e-84   pass


[higher_rank.txt]  147 lines  →  output/higher_rank.txt

verdict
  block m=8, all 16 ranks within 1.0e-50  : pass
  scalar c_00 m=16, all 16 ranks within 1.0e-50 : pass
  scalar c_11 m=16, all 16 ranks within 1.0e-50 : pass
  non-degener

In [37]:
#!/usr/bin/env python3
"""
eq24_comparison.py

Tests eq. (24) both ways in one run:

  case 1: leading term A_1(t+1)  ->  eq. (24) as printed   ->  should fail
  case 2: leading term A_1(t+2)  ->  eq. (24) corrected    ->  should pass

outputs:
  - full diagnostic trace for each case
  - fig2_eq24_printed.png
  - fig2_eq24_corrected.png
  - side-by-side summary table
"""

# imports and numerical precision
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpmath import mp

mp.dps = 100


# mock data (eqs. 71-72)
def build_mock(Nt=32):
    r, Ns = 2, 16
    E, Zt, Z = [], [], []
    for k in range(Ns):
        Ek = mp.mpf("0.1") * (k + 1)
        if k % 2 == 0:
            z0 = mp.mpf("1.0")
            z1 = mp.mpf("-0.1") if k == 0 else mp.mpf("0.1")
        else:
            z0 = mp.mpf("0.1")
            z1 = mp.mpf("1.0")
        Zt.append((z0, z1))
        Z.append((z0 / mp.sqrt(2 * Ek), z1 / mp.sqrt(2 * Ek)))
        E.append(Ek)

    C = []
    for t in range(Nt):
        Ct = mp.matrix(r, r)
        for k in range(Ns):
            ek = mp.exp(-E[k] * t)
            for a in range(r):
                for b in range(r):
                    Ct[a, b] += Z[k][a] * Z[k][b] * ek
        C.append(Ct)
    return C, E, Zt, Z


# matrix factorization helpers
def eig_values_vectors(A):
    E, ER = mp.eig(A)
    try:
        vals = [E[k, 0] for k in range(E.rows)]
    except (TypeError, AttributeError):
        vals = list(E)
    if not hasattr(ER, "rows"):
        ER = mp.matrix(ER)
    return vals, ER


def factor_delta(Delta):
    r = Delta.rows
    if r == 1:
        s = mp.sqrt(Delta[0, 0])
        return mp.matrix([[s]]), mp.matrix([[s]])
    vals, V = eig_values_vectors(Delta)
    Vinv = mp.inverse(V)
    S = mp.diag([mp.sqrt(vals[k]) for k in range(r)])
    return V * S, S * Vinv


# block lanczos recursion; `corrected` toggles eq. 24 base case only
def block_lanczos(C, m, corrected=True):
    Nt = len(C)
    r = C[0].rows
    alphas = [None] * (m + 1)
    betas = [None] * (m + 1)
    gammas = [None] * (m + 1)
    As = [None] * (m + 1)
    Gs = [None] * (m + 1)
    Bs = [None] * (m + 1)

    b1, g1 = factor_delta(C[0])
    betas[1], gammas[1] = b1, g1
    ib1, ig1 = mp.inverse(b1), mp.inverse(g1)
    A1 = [ib1 * C[t] * ig1 for t in range(Nt)]
    As[1] = A1
    alphas[1] = A1[1]

    if m == 1:
        return dict(alpha=alphas, beta=betas, gamma=gammas,
                    A=As, G=Gs, B=Bs)

    a1 = alphas[1]
    Delta2 = A1[2] - a1 * a1
    b2, g2 = factor_delta(Delta2)
    betas[2], gammas[2] = b2, g2
    ib2, ig2 = mp.inverse(b2), mp.inverse(g2)

    Gs[2] = [ib2 * (A1[t + 1] - a1 * A1[t]) for t in range(Nt - 1)]
    Bs[2] = [(A1[t + 1] - A1[t] * a1) * ig2 for t in range(Nt - 1)]

    if corrected:
        # leading term A_1(t+2)
        As[2] = [ib2 * (A1[t + 2] + a1 * A1[t] * a1
                        - (a1 * A1[t + 1] + A1[t + 1] * a1)) * ig2
                 for t in range(Nt - 2)]
    else:
        # leading term A_1(t+1), as printed
        As[2] = [ib2 * (A1[t + 1] + a1 * A1[t] * a1
                        - (a1 * A1[t + 1] + A1[t + 1] * a1)) * ig2
                 for t in range(Nt - 1)]
    alphas[2] = As[2][1]

    for j in range(2, m):
        Aj, Gj, Bj, Ajm1 = As[j], Gs[j], Bs[j], As[j - 1]
        aj, bj, gj = alphas[j], betas[j], gammas[j]

        Delta = Aj[2] - aj * aj - gj * bj
        bn, gn = factor_delta(Delta)
        betas[j + 1], gammas[j + 1] = bn, gn
        ibn, ign = mp.inverse(bn), mp.inverse(gn)

        Gs[j + 1] = [ibn * (Aj[t + 1] - aj * Aj[t] - gj * Bj[t])
                     for t in range(min(len(Aj) - 1, len(Bj)))]
        Bs[j + 1] = [(Aj[t + 1] - Aj[t] * aj - Gj[t] * bj) * ign
                     for t in range(min(len(Aj) - 1, len(Gj)))]

        LA = min(len(Aj) - 2, len(Ajm1), len(Bj) - 1, len(Gj) - 1)
        As[j + 1] = []
        for t in range(LA):
            term = (Aj[t + 2]
                    - (aj * Aj[t + 1] + Aj[t + 1] * aj)
                    + aj * Aj[t] * aj
                    + gj * Ajm1[t] * bj
                    - (gj * Bj[t + 1] + Gj[t + 1] * bj)
                    + gj * Bj[t] * aj
                    + aj * Gj[t] * bj)
            As[j + 1].append(ibn * term * ign)
        alphas[j + 1] = As[j + 1][1]

    return dict(alpha=alphas, beta=betas, gamma=gammas, A=As, G=Gs, B=Bs)


# tridiagonal assembly, spectral extraction, reconstruction
def assemble_T(alphas, betas, gammas, m, r):
    T = mp.matrix(m * r, m * r)
    for i in range(m):
        for a in range(r):
            for b in range(r):
                T[i * r + a, i * r + b] = alphas[i + 1][a, b]
    for i in range(m - 1):
        for a in range(r):
            for b in range(r):
                T[i * r + a, (i + 1) * r + b] = betas[i + 2][a, b]
                T[(i + 1) * r + a, i * r + b] = gammas[i + 2][a, b]
    return T


def submatrix(T, n):
    return mp.matrix([[T[i, j] for j in range(n)] for i in range(n)])


def ritz_energies(T):
    # the abs/log guards keep the "as printed" case from crashing on
    # spurious zero or sign-flipped eigenvalues during the diagnostic run
    vals, _ = eig_values_vectors(T)
    out = []
    for v in vals:
        if abs(v) > mp.mpf("1e-50"):
            try:
                out.append(-mp.log(v))
            except Exception:
                pass
    out.sort(key=lambda x: mp.re(x))
    return out


def reconstruct_C(T, b1, g1, Nt, r):
    out, P = [], mp.eye(T.rows)
    for _ in range(Nt):
        block = mp.matrix([[P[a, b] for b in range(r)] for a in range(r)])
        out.append(b1 * block * g1)
        P = P * T
    return out


def maxdiff(A, B, r):
    return max(abs(A[a, b] - B[a, b]) for a in range(r) for b in range(r))


def extract_hist(C, mmax, r, corrected=True):
    out = block_lanczos(C, mmax, corrected=corrected)
    T_full = assemble_T(out["alpha"], out["beta"],
                        out["gamma"], mmax, r)
    hist = []
    for m in range(1, mmax + 1):
        T_m = submatrix(T_full, m * r)
        E_vals = ritz_energies(T_m)
        hist.append(dict(m=m, t=2 * m - 1, energies=E_vals))
    return hist


# per-case diagnostics
def run_diagnostics(corrected):
    tag = "corrected (A_1(t+2))" if corrected else "as printed (A_1(t+1))"
    C, E, _Zt, _Z = build_mock(32)
    r = 2
    out8 = block_lanczos(C, 8, corrected=corrected)

    print()
    print(f"eq. (24) {tag}")

    # d1: biorthonormality at t=0
    a2_err = maxdiff(out8["A"][2][0], mp.eye(r), r)
    print()
    print("[d1] ||A_2(0) - I||  (should be zero by biorthonormality)")
    print(f"     ||A_2(0) - I||_max = {mp.nstr(a2_err, 5)}")
    print(f"     >>> {'pass: working precision' if float(a2_err) < 1e-3 else 'fail: order-one error'}")

    # d2: eq. 30 identity
    print()
    print("[d2] eq. (30): G_j(1) = gamma_j,  B_j(1) = beta_j")
    worst = mp.mpf(0)
    for j in range(2, 9):
        eg = maxdiff(out8["G"][j][1], out8["gamma"][j], r)
        eb = maxdiff(out8["B"][j][1], out8["beta"][j], r)
        worst = max(worst, eg, eb)
        print(f"     j={j}:  |G-gamma|={mp.nstr(eg, 3):<10}  |B-beta|={mp.nstr(eb, 3):<10}")
    print(f"     worst = {mp.nstr(worst, 5)}")

    # d3: eq. 60 reconstruction
    print()
    print("[d3] eq. (60): C(t) = beta_1 [T^t]_{11} gamma_1,  t <= 15")
    T8 = assemble_T(out8["alpha"], out8["beta"], out8["gamma"], 8, r)
    Crec = reconstruct_C(T8, out8["beta"][1], out8["gamma"][1], 32, r)
    print("     t   ||C_recon(t) - C(t)||_max")
    rec_errs = []
    for t in range(16):
        d = maxdiff(Crec[t], C[t], r)
        rec_errs.append(d)
        flag = "" if float(d) < 1e-20 else "   <<< fail"
        print(f"     {t:2d}   {mp.nstr(d, 5)}{flag}")

    # d4: block spectrum at m=8
    print()
    print("[d4] block lanczos spectrum at m=8  (target 0.1, 0.2, ..., 1.6)")
    Er = ritz_energies(T8)
    fail_count = 0
    for k in range(min(16, len(Er))):
        E_re = mp.re(Er[k])
        err = abs(E_re - E[k])
        if float(err) > 0.01:
            fail_count += 1
            print(f"     {k:2d}   E = {mp.nstr(E_re, 20):<24}  "
                  f"true = {mp.nstr(E[k], 4):<6}  "
                  f"|err| = {mp.nstr(err, 3):<10}   <<< fail")
        else:
            print(f"     {k:2d}   E = {mp.nstr(E_re, 20):<24}  "
                  f"true = {mp.nstr(E[k], 4):<6}  "
                  f"|err| = {mp.nstr(err, 3)}")
    if fail_count:
        print(f"     >>> {fail_count} of 16 ranks off by > 0.01")
    else:
        print("     >>> all 16 ranks within 0.01")

    return dict(
        corrected=corrected,
        a2_err=a2_err,
        worst=worst,
        rec_err_t15=rec_errs[15],
        fail_count=fail_count,
    )


# per-case figure
def plot_case(corrected, png):
    tag = "corrected  (A$_1$(t+2))" if corrected else "as printed  (A$_1$(t+1))"
    subtitle = ("reproduces fig. 2: block m=8, scalar m=16 "
                "converge exactly to black reference lines"
                if corrected else
                "fails: curves do not converge to black reference lines")

    C, _E, _Zt, _Z = build_mock(32)
    block_hist = extract_hist(C, 8, 2, corrected=corrected)
    c00_hist = extract_hist(
        [mp.matrix([[C[t][0, 0]]]) for t in range(32)], 16, 1, corrected=corrected)
    c11_hist = extract_hist(
        [mp.matrix([[C[t][1, 1]]]) for t in range(32)], 16, 1, corrected=corrected)

    fig, ax = plt.subplots(figsize=(10, 5))
    for k in range(16):
        ax.axhline(0.1 * (k + 1), color="k", lw=0.5, alpha=0.3)

    for hist, color, label in [
        (block_hist, "tab:blue", "block"),
        (c00_hist, "tab:orange", r"scalar $C_{00}$"),
        (c11_hist, "tab:red", r"scalar $C_{11}$"),
    ]:
        xs0, ys0, xs1, ys1 = [], [], [], []
        for e in hist:
            t = e["t"]
            Evals = sorted([mp.re(v) for v in e["energies"]])
            if len(Evals) >= 1 and -2 < Evals[0] < 5:
                xs0.append(t)
                ys0.append(float(Evals[0]))
            if len(Evals) >= 2 and -2 < Evals[1] < 5:
                xs1.append(t)
                ys1.append(float(Evals[1]))
        if xs0:
            ax.plot(xs0, ys0, "o-", color=color, lw=1.5, ms=4,
                    label=label + " rank 0")
        if xs1:
            ax.plot(xs1, ys1, "s--", color=color, lw=1.2, ms=3,
                    label=label + " rank 1")

    ax.set_xlabel(r"$t = 2m - 1$", fontsize=12)
    ax.set_ylabel(r"$E_k^{(m)} = -\log \lambda_k^{(m)}$", fontsize=12)
    ax.set_title(f"eq. (24) {tag}\n{subtitle}", fontsize=11)
    ax.set_xlim(0, 32)
    ax.set_ylim(-2, 5)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.savefig(png, dpi=150)
    plt.close(fig)
    print(f"     saved {png}")


# main entry point
def run():
    print("eq. (24) -- as printed vs corrected")

    s_printed = run_diagnostics(corrected=False)
    print()
    print("[plot] fig. 2 style, eq. (24) as printed")
    plot_case(corrected=False,
              png="fig2_eq24_printed.png")

    s_corrected = run_diagnostics(corrected=True)
    print()
    print("[plot] fig. 2 style, eq. (24) corrected")
    plot_case(corrected=True,
              png="fig2_eq24_corrected.png")

    print()
    print("side-by-side summary")
    print(f"  {'quantity':<28} {'printed':>22} {'corrected':>22}")
    print(f"  {'||A_2(0) - I||':<28} "
          f"{mp.nstr(s_printed['a2_err'], 5):>22} "
          f"{mp.nstr(s_corrected['a2_err'], 5):>22}")
    print(f"  {'eq. 30 worst':<28} "
          f"{mp.nstr(s_printed['worst'], 5):>22} "
          f"{mp.nstr(s_corrected['worst'], 5):>22}")
    print(f"  {'eq. 60 err at t=15':<28} "
          f"{mp.nstr(s_printed['rec_err_t15'], 5):>22} "
          f"{mp.nstr(s_corrected['rec_err_t15'], 5):>22}")
    print(f"  {'spectrum failures (of 16)':<28} "
          f"{s_printed['fail_count']:>22} "
          f"{s_corrected['fail_count']:>22}")
    print()
    vp = "fail" if (s_printed["fail_count"] > 0 or
                     float(s_printed["a2_err"]) > 1e-3) else "pass"
    vc = "fail" if (s_corrected["fail_count"] > 0 or
                    float(s_corrected["a2_err"]) > 1e-3) else "pass"
    print(f"  verdict  printed   : {vp}")
    print(f"  verdict  corrected : {vc}")


if __name__ == "__main__":
    run()

eq. (24) -- as printed vs corrected

eq. (24) as printed (A_1(t+1))

[d1] ||A_2(0) - I||  (should be zero by biorthonormality)
     ||A_2(0) - I||_max = 4.4684
     >>> fail: order-one error

[d2] eq. (30): G_j(1) = gamma_j,  B_j(1) = beta_j
     j=2:  |G-gamma|=5.36e-102   |B-beta|=5.36e-102 
     j=3:  |G-gamma|=2.86e-101   |B-beta|=2.86e-101 
     j=4:  |G-gamma|=5.34e-100   |B-beta|=5.34e-100 
     j=5:  |G-gamma|=4.07e-100   |B-beta|=4.11e-100 
     j=6:  |G-gamma|=5.71e-101   |B-beta|=5.71e-101 
     j=7:  |G-gamma|=2.68e-100   |B-beta|=2.68e-100 
     j=8:  |G-gamma|=2.86e-100   |B-beta|=2.89e-100 
     worst = 5.3399e-100

[d3] eq. (60): C(t) = beta_1 [T^t]_{11} gamma_1,  t <= 15
     t   ||C_recon(t) - C(t)||_max
      0   5.7149e-101
      1   5.7149e-101
      2   4.0183e-102
      3   0.9895   <<< fail
      4   2.1274   <<< fail
      5   3.1882   <<< fail
      6   4.3085   <<< fail
      7   5.5925   <<< fail
      8   7.083   <<< fail
      9   8.8285   <<< fail
     10